# 09. Save Final Model

In this notebook I will prepare the final training data and later save the final salary prediction model.

For now, I am doing only the first part:

1. Import libraries
2. Load cleaned dataset
3. Apply final training filters
4. Learn rare category mapping

## 1. Import Libraries

I import the libraries needed for loading data and preparing the final preprocessing rules.

In [21]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## 2. Load Cleaned Dataset

I load the cleaned salary dataset created earlier. This is the dataset I use as the starting point before final model training.

In [22]:
data_path = Path("../data/processed/cleaned_salary_data.csv")

df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (565, 9)


,work_year,experience_level,employment_type,job_title,employee_residence,remote_ratio,company_location,company_size,salary_in_usd
0,2020,MI,FT,Data Scientist,DE,0,DE,L,79833
1,2020,SE,FT,Machine Learning Scientist,JP,0,JP,S,260000
2,2020,SE,FT,Big Data Engineer,GB,50,GB,M,109024
3,2020,MI,FT,Product Data Analyst,HN,0,HN,S,20000
4,2020,SE,FT,Machine Learning Engineer,US,50,US,L,150000


## 3. Apply Final Training Filters

From the model improvement notebook, I decided to remove high salary outliers above 300000 USD.

I am not removing low salary rows because that did not improve the model.

In [23]:
high_salary_limit = 300000

df_final_training = df[df["salary_in_usd"] <= high_salary_limit].copy()

print(f"Original dataset shape: {df.shape}")
print(f"Final training dataset shape: {df_final_training.shape}")
print(f"Rows removed: {df.shape[0] - df_final_training.shape[0]}")

display(df_final_training["salary_in_usd"].describe())

Original dataset shape: (565, 9)
Final training dataset shape: (555, 9)
Rows removed: 10


count       555.000000
mean     105062.781982
std       58981.192674
min        2859.000000
25%       60000.000000
50%      100000.000000
75%      145500.000000
max      276000.000000
Name: salary_in_usd, dtype: float64

## 4. Learn Rare Category Mapping

I need to save the rare category logic because the future app must treat new inputs the same way as training data.

In the improvement notebook, I grouped rare values in these columns:

- `job_title`
- `employee_residence`
- `company_location`

Rule: if a category appears fewer than 5 times, I will map it to `Other`.

In [24]:
rare_category_columns = [
    "job_title",
    "employee_residence",
    "company_location"
]

rare_threshold = 5

rare_category_mapping = {}
for column in rare_category_columns:
    category_counts = df_final_training[column].value_counts()
    rare_categories = category_counts[category_counts < rare_threshold].index.tolist()
    kept_categories = category_counts[category_counts >= rare_threshold].index.tolist()

    rare_category_mapping[column] = {
        "rare_threshold": rare_threshold,
        "rare_categories": rare_categories,
        "kept_categories": kept_categories
    }

    print(f"Column: {column}")
    print(f"Total unique values: {category_counts.shape[0]}")
    print(f"Kept categories: {len(kept_categories)}")
    print(f"Rare categories mapped to Other: {len(rare_categories)}")
    print()

rare_category_mapping

Column: job_title
Total unique values: 49
Kept categories: 21
Rare categories mapped to Other: 28

Column: employee_residence
Total unique values: 57
Kept categories: 13
Rare categories mapped to Other: 44

Column: company_location
Total unique values: 50
Kept categories: 9
Rare categories mapped to Other: 41



{'job_title': {'rare_threshold': 5,
  'rare_categories': ['Data Analytics Engineer',
   'Applied Data Scientist',
   'Head of Data Science',
   'Analytics Engineer',
   'Lead Data Scientist',
   'Lead Data Analyst',
   'Machine Learning Infrastructure Engineer',
   'Computer Vision Software Engineer',
   'Data Science Engineer',
   'Machine Learning Developer',
   'Applied Machine Learning Scientist',
   'Product Data Analyst',
   'Cloud Data Engineer',
   'Director of Data Engineering',
   'Principal Data Engineer',
   'Principal Data Analyst',
   'Machine Learning Manager',
   '3D Computer Vision Researcher',
   'Marketing Data Analyst',
   'Data Specialist',
   'Finance Data Analyst',
   'Big Data Architect',
   'Staff Data Scientist',
   'ETL Developer',
   'Head of Machine Learning',
   'NLP Engineer',
   'Lead Machine Learning Engineer',
   'Financial Data Analyst'],
  'kept_categories': ['Data Scientist',
   'Data Engineer',
   'Data Analyst',
   'Machine Learning Engineer',
   

## 5. Group Rare Categories

Now I apply the rare category mapping I learned in the previous step.

For each selected categorical column, if the value is rare, I replace it with `Other`.

This keeps the training logic consistent with the improvement notebook.

In [25]:
df_final_grouped = df_final_training.copy()

for column in rare_category_columns:
    rare_categories = rare_category_mapping[column]["rare_categories"]
    df_final_grouped[column] = df_final_grouped[column].where(
        ~df_final_grouped[column].isin(rare_categories),
        "Other"
    )

    print(f"Column: {column}")
    print(f"Unique values before grouping: {df_final_training[column].nunique()}")
    print(f"Unique values after grouping: {df_final_grouped[column].nunique()}")
    print()

df_final_grouped

Column: job_title
Unique values before grouping: 49
Unique values after grouping: 22

Column: employee_residence
Unique values before grouping: 57
Unique values after grouping: 14

Column: company_location
Unique values before grouping: 50
Unique values after grouping: 10



,work_year,experience_level,employment_type,job_title,employee_residence,remote_ratio,company_location,company_size,salary_in_usd
0,2020,MI,FT,Data Scientist,DE,0,DE,L,79833
1,2020,SE,FT,Machine Learning Scientist,JP,0,JP,S,260000
2,2020,SE,FT,Big Data Engineer,GB,50,GB,M,109024
3,2020,MI,FT,Other,Other,0,Other,S,20000
4,2020,SE,FT,Machine Learning Engineer,US,50,US,L,150000
...,...,...,...,...,...,...,...,...,...
560,2022,SE,FT,Data Engineer,US,100,US,M,154000
561,2022,SE,FT,Data Engineer,US,100,US,M,126000
562,2022,SE,FT,Data Analyst,US,0,US,M,129000
563,2022,SE,FT,Data Analyst,US,100,US,M,150000


## 6. Train Final Ridge Pipeline

Now I train the final model using the exact decisions from the improvement notebook.

Final training choices:
- high salary outliers above 300000 USD removed
- rare categories grouped into `Other`
- numeric columns scaled
- categorical columns one-hot encoded
- Ridge Regression with `alpha=10`

This creates the final pipeline that I will save later.

In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [27]:
target_column = "salary_in_usd"

numeric_columns = [
    "work_year",
    "remote_ratio"
]

categorical_columns = [
    "experience_level",
    "employment_type",
    "job_title",
    "employee_residence",
    "company_location",
    "company_size"
]

X = df_final_grouped.drop(columns=[target_column])
y = df_final_grouped[target_column]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (555, 8)
y shape: (555,)


In [28]:
final_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns)
    ],
    remainder="drop"
)

final_model = Pipeline(
    steps=[
        ("preprocessor", final_preprocessor),
        ("model", Ridge(alpha=10.0))
    ]
)

final_model.fit(X, y)

print("Final Ridge pipeline trained successfully.")

Final Ridge pipeline trained successfully.


In [29]:
import joblib

In [30]:
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

model_path = models_dir / "salary_prediction_ridge_pipeline.joblib"

joblib.dump(final_model, model_path)

print(f"Model saved to: {model_path}")

Model saved to: ../models/salary_prediction_ridge_pipeline.joblib


## 8. Save Metadata

Now I save the training rules used before model prediction.

This metadata is important because the future app must apply the same rare-category grouping before sending data into the model.

The model pipeline handles scaling and one-hot encoding, but it does not automatically know my rare-category mapping rules unless I save them separately.

In [31]:
import json
metadata = {
    "model_name": "salary_prediction_ridge_pipeline",
    "model_type": "Ridge Regression",
    "ridge_alpha": 10.0,
    "target_column": target_column,
    "input_columns": X.columns.tolist(),
    "numeric_columns": numeric_columns,
    "categorical_columns": categorical_columns,
    "high_salary_limit": high_salary_limit,
    "rare_threshold": rare_threshold,
    "rare_category_columns": rare_category_columns,
    "rare_category_mapping": rare_category_mapping
}

metadata_path = models_dir / "salary_prediction_metadata.json"

with open(metadata_path, "w") as file:
    json.dump(metadata, file, indent=4)

print(f"Metadata saved to: {metadata_path}")

Metadata saved to: ../models/salary_prediction_metadata.json


## 9. Test Loaded Model

Now I test the saved model and saved metadata.

This is important because saving is only useful if I can load the files again and make a prediction successfully.

In [32]:
loaded_model = joblib.load(model_path)

with open(metadata_path, "r") as file:
    loaded_metadata = json.load(file)

print("Model and metadata loaded successfully.")
print(loaded_metadata["model_name"])

Model and metadata loaded successfully.
salary_prediction_ridge_pipeline


In [33]:
def apply_rare_category_mapping(input_data, metadata):
    input_data = input_data.copy()

    for column in metadata["rare_category_columns"]:
        kept_categories = metadata["rare_category_mapping"][column]["kept_categories"]

        input_data[column] = input_data[column].where(
            input_data[column].isin(kept_categories),
            "Other"
        )

    return input_data

In [34]:
sample_input = X.iloc[[0]].copy()

print("Original sample input:")
display(sample_input)

sample_input_grouped = apply_rare_category_mapping(sample_input, loaded_metadata)

print("Sample input after rare category grouping:")
display(sample_input_grouped)

sample_prediction = loaded_model.predict(sample_input_grouped)[0]

print(f"Predicted salary: {sample_prediction:.2f}")

Original sample input:


,work_year,experience_level,employment_type,job_title,employee_residence,remote_ratio,company_location,company_size
0,2020,MI,FT,Data Scientist,DE,0,DE,L


Sample input after rare category grouping:


,work_year,experience_level,employment_type,job_title,employee_residence,remote_ratio,company_location,company_size
0,2020,MI,FT,Data Scientist,DE,0,DE,L


Predicted salary: 71161.46
